# Viscan OPG — benchmark ladder: can we beat the measured Faster R-CNN result?

One Kaggle GPU session that measures the full pre-registered ladder from `RESEARCH_BENCHMARKS.md` §7:

| id | pipeline | what is new vs the deployed model |
|---|---|---|
| P0 | Faster R-CNN (downloaded, SHA-256 verified) | reference row; must reproduce the published 0.496 / 0.329 |
| P1 | + horizontal-flip TTA | 2x forward passes |
| P2 | + native-scale sliced inference (800x800 tiles, WBF with the full-image pass) | attacks the 0.474x mean downscale of DENTEX images |
| P3 | P2 + RetinaNet R50-FPN-V2 trained here, WBF over all members | architecture diversity |
| P4 | P2 + ConvNeXt-Tiny crop refiner (`rescore`) | attacks deep_caries->caries confusion and caries background FPs |
| P5 | P3 + crop refiner (`rescore`) | |
| P6 | P2 + crop refiner (`reclass`) | drops boxes the classifier calls background |
| P7 | P3 + crop refiner (`reclass`) | |

**Protocol (fixed before the run)**
- DENTEX pinned revision, `SEED=42` split: 635 train / 70 val / 50 held-out test.
- Every pipeline is fused with WBF at IoU 0.55; the operating threshold is selected on the **70-image validation split only** (macro-F1 sweep 0.10-0.90), then applied once to the test split.
- Pipeline choice uses **validation mAP@50:95 only**. All 8 test rows are reported so no result is hidden.
- The crop refiner is trained only on train-split GT crops.
- No number below is pre-filled: every metric is read from the JSON this run writes. If a stage loses, its row stays in the table.

**Before running:** Settings -> Accelerator **T4 (x1 is enough; x2 is fine)**, Settings -> Internet **On**.
RetinaNet training (12 epochs) is the long step; sliced inference adds ~10 forward passes per image.

In [1]:
import os, sys, json, time, platform, subprocess
from pathlib import Path

import torch, torchvision

print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("platform   :", platform.platform())

assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> T4, then re-run."
print("gpu        :", torch.cuda.get_device_name(0))
print("vram (GB)  :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

python     : 3.13.15
torch      : 2.11.0+cu128
torchvision: 0.26.0+cu128
platform   : Linux-6.18.55+-x86_64-with-glibc2.39
gpu        : Tesla T4
vram (GB)  : 14.6


In [2]:
REPO_URL = "https://github.com/surendra902/viscan-opg.git"
PINNED_COMMIT = "1ac108065852d42ba66bf799fc7d4b443ff49ec2"
WORKDIR = Path("/kaggle/working/viscan-opg")

if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "--quiet", "origin"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(WORKDIR)], check=True)

subprocess.run(["git", "-C", str(WORKDIR), "checkout", "--quiet", PINNED_COMMIT], check=True)
head = subprocess.run(["git", "-C", str(WORKDIR), "rev-parse", "HEAD"],
                      capture_output=True, text=True, check=True).stdout.strip()
assert head == PINNED_COMMIT, f"checkout mismatch: {head}"
print("repo pinned at", head)

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))

for mod in ("yaml", "pycocotools", "matplotlib", "pandas", "PIL"):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", mod], check=True)

import model as viscan_model
assert viscan_model.HAS_TORCHVISION
print("imports OK | classes:", viscan_model.CLASS_NAMES)

repo pinned at 1ac108065852d42ba66bf799fc7d4b443ff49ec2
imports OK | classes: {0: 'background', 1: 'impacted_tooth', 2: 'caries', 3: 'periapical_lesion', 4: 'deep_caries'}


In [3]:
os.chdir(WORKDIR)

data_dir = Path("data/dentex")
test_images = data_dir / "images" / "test"
ready = ((data_dir / "annotations" / "instances_test.json").exists()
         and test_images.is_dir() and any(test_images.iterdir()))

if ready:
    print("[SKIP] dataset already prepared at", data_dir.resolve())
else:
    print("[RUN] downloading + converting DENTEX (pinned revision) into", data_dir.resolve())
    proc = subprocess.run(["bash", "-lc",
        f'set -o pipefail; "{sys.executable}" prepare_data.py data/dentex 2>&1 | tee prepare_data.log'],
        check=False)
    assert proc.returncode == 0, "prepare_data.py failed - check prepare_data.log"

[RUN] downloading + converting DENTEX (pinned revision) into /kaggle/working/viscan-opg/data/dentex
[DOWNLOAD] Downloading Validation Annotations JSON from https://huggingface.co/datasets/ibrahimhamamci/DENTEX/resolve/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX/validation_triple.json...
  Progress: 101% (0MB / 0MB)
  [DONE] Download complete.
[DOWNLOAD] Downloading Training Images ZIP from https://huggingface.co/datasets/ibrahimhamamci/DENTEX/resolve/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX/training_data.zip...
  Progress: 100% (10421MB / 10421MB)
  [DONE] Download complete.
[DOWNLOAD] Downloading Validation Images ZIP from https://huggingface.co/datasets/ibrahimhamamci/DENTEX/resolve/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX/validation_data.zip...
  Progress: 100% (142MB / 142MB)
  [DONE] Download complete.
[EXTRACT] Extracting training_data.zip to data/dentex/extracted_train...
  [DONE] Extracted 708 files.
[EXTRACT] Extracting validation_data.zip to data/dentex/extr

In [4]:
os.chdir(WORKDIR)
import pandas as pd

data_dir = Path("data/dentex")
summary = {}
for s in ("train", "val", "test"):
    coco = json.loads((data_dir / "annotations" / f"instances_{s}.json").read_text(encoding="utf-8"))
    per_class = {cid: 0 for cid in viscan_model.CLASS_NAMES if cid != 0}
    for ann in coco["annotations"]:
        per_class[ann["category_id"]] += 1
    summary[s] = {
        "images": len(coco["images"]),
        "boxes": len(coco["annotations"]),
        "per_class": per_class,
        "files": {img["file_name"] for img in coco["images"]},
        "scales": [min(800.0 / min(i["width"], i["height"]), 1333.0 / max(i["width"], i["height"]))
                   for i in coco["images"]],
    }

rows = []
for s, d in summary.items():
    row = {"split": s, "images": d["images"], "boxes": d["boxes"]}
    row.update({viscan_model.CLASS_NAMES[cid]: n for cid, n in d["per_class"].items()})
    row["detector scale mean"] = round(sum(d["scales"]) / len(d["scales"]), 3)
    rows.append(row)
display(pd.DataFrame(rows).set_index("split"))

assert summary["train"]["images"] == 635 and summary["val"]["images"] == 70
assert summary["test"]["images"] == 50
assert summary["train"]["boxes"] + summary["val"]["boxes"] == 3529
assert summary["test"]["boxes"] == 182
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not (summary[a]["files"] & summary[b]["files"]), f"{a}/{b} image overlap"
print("split verified: 635/70/50 images, 3529/182 boxes, no overlap")

,images,boxes,impacted_tooth,caries,periapical_lesion,deep_caries,detector scale mean
split,,,,,,,
train,635,3190,545,2004,144,497,0.474
val,70,339,59,185,14,81,0.470
test,50,182,40,101,9,32,0.481


split verified: 635/70/50 images, 3529/182 boxes, no overlap


In [5]:
os.chdir(WORKDIR)
import hashlib
from urllib.request import urlretrieve

WEIGHTS_URL = "https://huggingface.co/surendra08/viscan-opg-weights/resolve/main/best_model.pth"
WEIGHTS_SHA256 = "e0b49b12f48fc71ec1f583db08f107650659571e7c2b9cbd454dbfed13b46a1f"

ckpt = Path("checkpoints/best_model.pth")
ckpt.parent.mkdir(parents=True, exist_ok=True)
if not ckpt.exists():
    print("[DOWNLOAD]", WEIGHTS_URL)
    urlretrieve(WEIGHTS_URL, ckpt)

digest = hashlib.sha256()
with open(ckpt, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        digest.update(chunk)
actual = digest.hexdigest()
assert actual == WEIGHTS_SHA256, f"checkpoint hash mismatch: {actual}"
print(f"Faster R-CNN checkpoint verified: {ckpt.stat().st_size:,} bytes | sha256 {actual}")

[DOWNLOAD] https://huggingface.co/surendra08/viscan-opg-weights/resolve/main/best_model.pth
Faster R-CNN checkpoint verified: 173,466,789 bytes | sha256 e0b49b12f48fc71ec1f583db08f107650659571e7c2b9cbd454dbfed13b46a1f


In [6]:
os.chdir(WORKDIR)

def run(cmd, log):
    proc = subprocess.run(["bash", "-lc", f'set -o pipefail; {cmd} 2>&1 | tee {log}'], check=False)
    assert proc.returncode == 0, f"command failed, inspect {log}"
    print("done ->", log)

# P0: deployed Faster R-CNN, no TTA. Dumps val + test predictions for fusion and re-scoring.
run(f'"{sys.executable}" evaluate.py --weights checkpoints/best_model.pth '
    f'--config configs/baseline.yaml --data_dir data/dentex --output_dir evaluation/P0 '
    f'--skip_variant --select_threshold val --dump_predictions preds/frcnn', "eval_p0.log")

assert Path("preds/frcnn/predictions_val.json").exists()
assert Path("preds/frcnn/predictions_test.json").exists()
print(json.loads(Path("evaluation/P0/final_evaluation_report.json").read_text())["coco_map"]["overall"])

[EVAL] Evaluating using device: cuda | TTA: none
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
[DUMP] Wrote test predictions to preds/frcnn/predictions_test.json

--- 1. COCO mAP Evaluation (fasterrcnn_resnet50_fpn_v2) ---
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.13s).
Accumulating evaluation results...
DONE (t=0.04s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.329
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.496
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.383
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.329
 Average Recall     (AR) @[ IoU=0.50:0.95 | ar

In [7]:
os.chdir(WORKDIR)

# P1 ablation: horizontal-flip TTA on the full image.
run(f'"{sys.executable}" evaluate.py --weights checkpoints/best_model.pth '
    f'--config configs/baseline.yaml --data_dir data/dentex --output_dir evaluation/P1 '
    f'--skip_variant --select_threshold val --tta hflip --dump_predictions preds/frcnn_tta',
    "eval_p1.log")

[EVAL] Evaluating using device: cuda | TTA: hflip
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
[DUMP] Wrote test predictions to preds/frcnn_tta/predictions_test.json

--- 1. COCO mAP Evaluation (fasterrcnn_resnet50_fpn_v2) ---
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.14s).
Accumulating evaluation results...
DONE (t=0.04s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.345
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.528
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.398
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.345
 Average Recall     (AR) @[ IoU=0.50:0.95

In [8]:
os.chdir(WORKDIR)

# P2: full image (hflip TTA) + native-scale 800x800 tiles, one dump per view group.
for split in ("val", "test"):
    run(f'"{sys.executable}" slice_inference.py --weights checkpoints/best_model.pth '
        f'--config configs/baseline.yaml --split {split} --data_dir data/dentex '
        f'--output_dir preds/slice --views both --tile 800 --overlap 0.25 '
        f'--tta hflip --tile_tta none --max_dets_tile 100', f"slice_{split}.log")
    meta = json.loads(Path(f"preds/slice/slice_meta_{split}.json").read_text())
    print(split, {k: meta[k] for k in ("num_images", "num_tiles", "tiles_per_image_mean",
                                      "detections", "ms_per_image")})

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
[SLICE] split=val images=70 device=cuda views=both
[SLICE] detections: {'full': 4841, 'tiles': 26115} | tiles: 796
[SLICE] wrote dumps to preds/slice
done -> slice_val.log
val {'num_images': 70, 'num_tiles': 796, 'tiles_per_image_mean': 11.37, 'detections': {'full': 4841, 'tiles': 26115}, 'ms_per_image': {'full': 437.44, 'tiles': 1719.71}}
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
[SLICE] split=test images=50 device=cuda views=both
[SLICE] detections: {'full': 3303, 'tiles': 17059} | tiles: 549
[SLICE] wrote dumps to preds/slice
done -> slice_test.log
test {'num_images': 50, 'num_tiles': 549, 'tiles_per_image_mean': 10.98, 'detections': {'full': 3303, 'tiles': 17059}, 'ms_per_image': {'full': 438.46, 'tiles': 1668.21}}


In [9]:
os.chdir(WORKDIR)

# RetinaNet on the identical split / seed / schedule / input scale. Longest step (~12 epochs).
t0 = time.time()
run(f'"{sys.executable}" train.py --config configs/retinanet.yaml', "train_retinanet.log")
print(f"retinanet training wall time: {(time.time() - t0) / 60.0:.1f} min")

ckpt = Path("checkpoints/retinanet/best_model.pth")
assert ckpt.exists(), "best RetinaNet checkpoint missing"
state = torch.load(ckpt, map_location="cpu", weights_only=False)
print("best epoch:", state["epoch"], "| best val mAP@50:", round(float(state["val_map50"]), 4))
del state
torch.cuda.empty_cache()

[INFO] Using training device: cuda
Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth" to /root/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth
100%|██████████| 146M/146M [00:00<00:00, 168MB/s]  
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
[INFO] AMP enabled: True
[INFO] LR scheduler: CosineAnnealingLR

[START] Beginning training for 12 epochs...
Epoch [1/12] Batch [50/318] Loss: 1.0117 lr: 0.005000
Epoch [1/12] Batch [100/318] Loss: 1.3593 lr: 0.005000
Epoch [1/12] Batch [150/318] Loss: 1.2323 lr: 0.005000
Epoch [1/12] Batch [200/318] Loss: 1.2614 lr: 0.005000
Epoch [1/12] Batch [250/318] Loss: 1.2765 lr: 0.005000
Epoch [1/12] Batch [300/318] Loss: 1.1707 lr: 0.005000
Epoch [1/12] Batch [318/318] Loss: 1.2296 lr: 0.005000

[EVAL] Running validation evaluation for Epoch 1...
Loading and preparing re

In [10]:
os.chdir(WORKDIR)

run(f'"{sys.executable}" evaluate.py --weights checkpoints/retinanet/best_model.pth '
    f'--config configs/retinanet.yaml --data_dir data/dentex --output_dir evaluation/retina '
    f'--skip_variant --select_threshold val --tta hflip --dump_predictions preds/retinanet_tta',
    "eval_retinanet.log")

[EVAL] Evaluating using device: cuda | TTA: hflip
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
[DUMP] Wrote test predictions to preds/retinanet_tta/predictions_test.json

--- 1. COCO mAP Evaluation (retinanet_resnet50_fpn_v2) ---
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.40s).
Accumulating evaluation results...
DONE (t=0.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.053
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.095
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.056
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.053
 Average Recall     (AR) @[ IoU=0.50:0

In [11]:
os.chdir(WORKDIR)

# Crop refiner: crop bank from train-split GT only, then 4 epochs of ConvNeXt-Tiny.
run(f'"{sys.executable}" cascade_refine.py build --data_dir data/dentex '
    f'--output_dir checkpoints/cascade', "cascade_build.log")
meta = json.loads(Path("checkpoints/cascade/crop_bank_meta.json").read_text())
assert meta["source_split"] == "train", "crop bank must come from the train split only"
print("crop bank:", meta["num_samples"], "crops |", meta["counts"], "| context", meta["context_range"])

run(f'"{sys.executable}" cascade_refine.py train --bank_dir checkpoints/cascade '
    f'--output checkpoints/cascade/convnext_cascade.pth --epochs 4 --batch_size 64', "cascade_train.log")
assert Path("checkpoints/cascade/convnext_cascade.pth").exists()

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
[CASCADE] crop bank: 6380 crops {'background': 3190, 'impacted_tooth': 545, 'caries': 2004, 'periapical_lesion': 144, 'deep_caries': 497} -> checkpoints/cascade
done -> cascade_build.log
crop bank: 6380 crops | {'background': 3190, 'impacted_tooth': 545, 'caries': 2004, 'periapical_lesion': 144, 'deep_caries': 497} | context [1.15, 1.45]
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth
100%|██████████| 109M/109M [00:00<00:00, 187MB/s]  
[CASCADE] epoch 1/4 loss=0.8579 acc=0.7112
[CASCADE] epoch 2/4 loss=0.5301 acc=0.821
[CASCADE] epoch 3/4 loss=0.3445 acc=0.8857
[CASCADE] epoch 4/4 loss=0.2548 acc=0.9206
[CASCADE] saved checkpoints/cascade/convnext_cascade.pth
done -> cascade_train.log


In [12]:
os.chdir(WORKDIR)

# Apply the refiner to each member dump. Separate output dirs keep file names unique.
SOURCES = [
    ("preds/slice", "full", "preds/cascade_slice_full"),
    ("preds/slice", "tiles", "preds/cascade_slice_tiles"),
    ("preds/retinanet_tta", None, "preds/cascade_retinanet"),
]

for mode in ("rescore", "reclass"):
    for src_dir, view, out_dir in SOURCES:
        for split in ("val", "test"):
            src = (f"{src_dir}/predictions_{split}_{view}.json" if view
                   else f"{src_dir}/predictions_{split}.json")
            run(f'"{sys.executable}" cascade_refine.py apply '
                f'--checkpoint checkpoints/cascade/convnext_cascade.pth --preds {src} '
                f'--split {split} --data_dir data/dentex --output_dir {out_dir} '
                f'--mode {mode} --max_dets 100', f"cascade_{mode}_{src_dir.split('/')[-1]}_{split}.log")

print("cascade dumps:")
for p in sorted(Path("preds").glob("cascade_*/cascade_meta_*.json")):
    m = json.loads(p.read_text())
    print(f"  {p.name}: {m['num_considered']}/{m['num_input']} -> {m['num_output']} "
          f"(dropped bg={m['dropped_background']}, relabelled={m['relabelled']})")

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
[CASCADE] rescore: 4841 -> 4841 dets (dropped bg=0, relabelled=0) -> preds/cascade_slice_full/predictions_val_rescore.json
done -> cascade_rescore_slice_val.log
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
[CASCADE] rescore: 3303 -> 3303 dets (dropped bg=0, relabelled=0) -> preds/cascade_slice_full/predictions_test_rescore.json
done -> cascade_rescore_slice_test.log
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
[CASCADE] rescore: 26115 -> 21206 dets (dropped bg=0, relabelled=0) -> preds/cascade_slice_tiles/predictions_val_rescore.json
done -> cascade_rescore_slice_val.log
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
[CASCADE] rescore: 17059 -> 14353 dets (dropped bg=0, relabelled=0) -> preds/cascade_slice_tiles/predictions_test_rescore.json
done -> cascade_rescore_slice_test.log
loading annota

In [13]:
os.chdir(WORKDIR)
import io, contextlib
import pandas as pd
from pycocotools.coco import COCO

from ensemble_wbf import run_ensemble, fuse_predictions
from evaluate import run_coco_map

TEST_GT = "data/dentex/annotations/instances_test.json"
VAL_GT = "data/dentex/annotations/instances_val.json"
TEST_IMAGES = "data/dentex/images/test"
coco_val = COCO(VAL_GT)

def _silent(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)

M = {
    "frcnn":               {"val": ["preds/frcnn/predictions_val.json"], "test": ["preds/frcnn/predictions_test.json"]},
    "frcnn_tta":           {"val": ["preds/frcnn_tta/predictions_val.json"], "test": ["preds/frcnn_tta/predictions_test.json"]},
    "slice":               {"val": ["preds/slice/predictions_val_full.json", "preds/slice/predictions_val_tiles.json"],
                            "test": ["preds/slice/predictions_test_full.json", "preds/slice/predictions_test_tiles.json"]},
    "retina":              {"val": ["preds/retinanet_tta/predictions_val.json"], "test": ["preds/retinanet_tta/predictions_test.json"]},
}
for view in ("full", "tiles"):
    for mode in ("rescore", "reclass"):
        key = f"casc_slice_{view}_{mode}"
        M[key] = {"val": [f"preds/cascade_slice_{view}/predictions_val_{mode}.json"],
                  "test": [f"preds/cascade_slice_{view}/predictions_test_{mode}.json"]}
for mode in ("rescore", "reclass"):
    M[f"casc_retina_{mode}"] = {"val": [f"preds/cascade_retinanet/predictions_val_{mode}.json"],
                                 "test": [f"preds/cascade_retinanet/predictions_test_{mode}.json"]}

for name, d in M.items():
    for split, paths in d.items():
        for p in paths:
            assert Path(p).exists(), f"missing dump {p}"

PIPELINES = [
    ("P0 Faster R-CNN",                    ["frcnn"], "p0"),
    ("P1 FRCNN + hflip TTA",               ["frcnn_tta"], "p1"),
    ("P2 FRCNN sliced (full+tiles)",       ["slice"], "p2"),
    ("P3 P2 + RetinaNet (WBF)",            ["slice", "retina"], "p3"),
    ("P4 P2 + cascade rescore",            ["casc_slice_full_rescore", "casc_slice_tiles_rescore"], "p4"),
    ("P5 P3 + cascade rescore",            ["casc_slice_full_rescore", "casc_slice_tiles_rescore",
                                            "casc_retina_rescore"], "p5"),
    ("P6 P2 + cascade reclass",            ["casc_slice_full_reclass", "casc_slice_tiles_reclass"], "p6"),
    ("P7 P3 + cascade reclass",            ["casc_slice_full_reclass", "casc_slice_tiles_reclass",
                                            "casc_retina_reclass"], "p7"),
]

def val_metrics(paths):
    fused, _ = fuse_predictions([json.loads(Path(p).read_text()) for p in paths])
    return _silent(run_coco_map, coco_val, fused)["overall"]

rows, reports = [], {}
for label, members, tag in PIPELINES:
    val_paths = [p for m in members for p in M[m]["val"]]
    test_paths = [p for m in members for p in M[m]["test"]]
    out_dir = f"evaluation/pipelines/{tag}"

    _silent(run_ensemble, pred_paths=test_paths, gt_path=TEST_GT, images_dir=TEST_IMAGES,
            val_pred_paths=val_paths, val_gt_path=VAL_GT, output_dir=out_dir, iou_thresh=0.55)

    rep = json.loads((Path(out_dir) / "final_evaluation_report.json").read_text())
    vm = val_metrics(val_paths)
    reports[label] = rep
    rows.append({
        "pipeline": label,
        "val AP": vm["map_50_95"], "val AP50": vm["map_50"],
        "test AP": rep["coco_map"]["overall"]["map_50_95"],
        "test AP50": rep["coco_map"]["overall"]["map_50"],
        "test AP75": rep["coco_map"]["overall"]["map_75"],
        "test AR@100": rep["coco_map"]["overall"].get("ar_100"),
        "tau": rep["threshold_selection"]["best_threshold"],
        "test macro-F1": rep["threshold_metrics"]["macro_f1"],
    })
    print(f"{label:38s} val AP={vm['map_50_95']:.4f}  val AP50={vm['map_50']:.4f}")

table = pd.DataFrame(rows).set_index("pipeline").round(4)
display(table)

# Selection uses validation mAP@50:95 only.
selected = table["val AP"].idxmax()
print(f"selected on validation AP: {selected}")

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
P0 Faster R-CNN                        val AP=0.3602  val AP50=0.5705
P1 FRCNN + hflip TTA                   val AP=0.3685  val AP50=0.5773
P2 FRCNN sliced (full+tiles)           val AP=0.1960  val AP50=0.3390
P3 P2 + RetinaNet (WBF)                val AP=0.1451  val AP50=0.2553
P4 P2 + cascade rescore                val AP=0.2238  val AP50=0.3908
P5 P3 + cascade rescore                val AP=0.1859  val AP50=0.3257
P6 P2 + cascade reclass                val AP=0.1379  val AP50=0.2476
P7 P3 + cascade reclass                val AP=0.0804  val AP50=0.1533


,val AP,val AP50,test AP,test AP50,test AP75,test AR@100,tau,test macro-F1
pipeline,,,,,,,,
P0 Faster R-CNN,0.3602,0.5705,0.3293,0.4961,0.3830,0.6834,0.70,0.4016
P1 FRCNN + hflip TTA,0.3685,0.5773,0.3446,0.5281,0.3978,0.6852,0.80,0.3988
P2 FRCNN sliced (full+tiles),0.1960,0.3390,0.2189,0.3557,0.2521,0.6638,0.35,0.2896
P3 P2 + RetinaNet (WBF),0.1451,0.2553,0.1790,0.3015,0.1965,0.6462,0.30,0.2661
P4 P2 + cascade rescore,0.2238,0.3908,0.2211,0.3589,0.2467,0.6560,0.40,0.3543
P5 P3 + cascade rescore,0.1859,0.3257,0.1936,0.3168,0.2084,0.6613,0.40,0.3172
P6 P2 + cascade reclass,0.1379,0.2476,0.1700,0.2715,0.1903,0.5573,0.25,0.3184
P7 P3 + cascade reclass,0.0804,0.1533,0.1437,0.2356,0.1659,0.6075,0.30,0.2320


selected on validation AP: P1 FRCNN + hflip TTA


In [14]:
os.chdir(WORKDIR)
import pandas as pd

# Reproduction check for P0 against the published report (soft: prints drift, does not abort).
PUBLISHED = {"AP50": 0.496, "AP": 0.329, "AP75": 0.383, "AR@100": 0.683}
p0 = table.loc["P0 Faster R-CNN"]
measured = {"AP50": p0["test AP50"], "AP": p0["test AP"], "AP75": p0["test AP75"], "AR@100": p0["test AR@100"]}
for k, v in measured.items():
    drift = float(v) - PUBLISHED[k]
    print(f"P0 {k:7s} measured={float(v):.4f}  published={PUBLISHED[k]:.3f}  drift={drift:+.4f}")

best = table.loc[selected]
base = table.loc["P0 Faster R-CNN"]
print("\nSelected pipeline:", selected)
print(f"  test AP50 {best['test AP50']:.4f} vs P0 {base['test AP50']:.4f} "
      f"({best['test AP50'] - base['test AP50']:+.4f})")
print(f"  test AP    {best['test AP']:.4f} vs P0 {base['test AP']:.4f} "
      f"({best['test AP'] - base['test AP']:+.4f})")
print(f"  vs published 0.496 / 0.329: AP50 {best['test AP50'] - 0.496:+.4f}, AP {best['test AP'] - 0.329:+.4f}")
print("  beats published AP50:", "YES" if best['test AP50'] > 0.496 else "NO",
      "| beats published AP:", "YES" if best['test AP'] > 0.329 else "NO")

P0 AP50    measured=0.4961  published=0.496  drift=+0.0001
P0 AP      measured=0.3293  published=0.329  drift=+0.0003
P0 AP75    measured=0.3830  published=0.383  drift=+0.0000
P0 AR@100  measured=0.6834  published=0.683  drift=+0.0004

Selected pipeline: P1 FRCNN + hflip TTA
  test AP50 0.5281 vs P0 0.4961 (+0.0320)
  test AP    0.3446 vs P0 0.3293 (+0.0153)
  vs published 0.496 / 0.329: AP50 +0.0321, AP +0.0156
  beats published AP50: YES | beats published AP: YES


In [15]:
os.chdir(WORKDIR)
import pandas as pd
import numpy as np

rep = reports[selected]
per_class = pd.DataFrame(rep["coco_map"]["per_class"]).T.round(4)
display(per_class)

cm = pd.DataFrame(np.asarray(rep["threshold_metrics"]["confusion_matrix"]),
                  index=["background"] + [viscan_model.CLASS_NAMES[i] for i in range(1, 5)],
                  columns=["background"] + [viscan_model.CLASS_NAMES[i] for i in range(1, 5)])
print(f"confusion matrix at tau={rep['threshold_selection']['best_threshold']:.2f} (rows=truth, cols=prediction):")
display(cm)

# External reference: DENTEX 2023 diagnosis leaderboard (250 hidden test images, percent as published).
EXTERNAL = [
    ("Hamamci I. (HierarchicalDet)", 0.691, 37.6, 60.2, 44.0),
    ("Mei S. (YOLOrtho)", 0.5972, 37.44, 56.72, 45.49),
    ("He L. (DINO + U-Net + YOLOv8 + WBF)", 0.5921, 37.06, 54.31, 44.77),
    ("Chen S. (DiffusionDet)", 0.644, 37.0, 58.1, 42.6),
    ("Choi K. (DETDet)", 0.7585, 36.48, 57.30, 42.84),
    ("Faster R-CNN reference", 0.533, 33.2, 54.3, 38.0),
    ("RetinaNet reference", 0.587, 32.5, 54.2, 35.6),
]
display(pd.DataFrame(EXTERNAL, columns=["entry", "AR@100", "AP", "AP50", "AP75"]).set_index("entry"))
print("Context only: different test images (250 hidden vs our 50 public), 705 vs 635 training images.")
print("Our rows in the same units:")
display(table[["test AR@100", "test AP", "test AP50", "test AP75"]].round(4) * 100)

,map_50_95,map_50
impacted_tooth,0.5653,0.8624
caries,0.2719,0.3887
periapical_lesion,0.2266,0.3759
deep_caries,0.3145,0.4852


confusion matrix at tau=0.80 (rows=truth, cols=prediction):


,background,impacted_tooth,caries,periapical_lesion,deep_caries
background,0,11,115,3,6
impacted_tooth,5,35,0,0,0
caries,44,1,54,0,2
periapical_lesion,5,0,2,1,1
deep_caries,15,0,11,0,6


,AR@100,AP,AP50,AP75
entry,,,,
Hamamci I. (HierarchicalDet),0.6910,37.60,60.20,44.00
Mei S. (YOLOrtho),0.5972,37.44,56.72,45.49
He L. (DINO + U-Net + YOLOv8 + WBF),0.5921,37.06,54.31,44.77
Chen S. (DiffusionDet),0.6440,37.00,58.10,42.60
Choi K. (DETDet),0.7585,36.48,57.30,42.84
Faster R-CNN reference,0.5330,33.20,54.30,38.00
RetinaNet reference,0.5870,32.50,54.20,35.60


Context only: different test images (250 hidden vs our 50 public), 705 vs 635 training images.
Our rows in the same units:


,test AR@100,test AP,test AP50,test AP75
pipeline,,,,
P0 Faster R-CNN,68.34,32.93,49.61,38.30
P1 FRCNN + hflip TTA,68.52,34.46,52.81,39.78
P2 FRCNN sliced (full+tiles),66.38,21.89,35.57,25.21
P3 P2 + RetinaNet (WBF),64.62,17.90,30.15,19.65
P4 P2 + cascade rescore,65.60,22.11,35.89,24.67
P5 P3 + cascade rescore,66.13,19.36,31.68,20.84
P6 P2 + cascade reclass,55.73,17.00,27.15,19.03
P7 P3 + cascade reclass,60.75,14.37,23.56,16.59


In [16]:
os.chdir(WORKDIR)
import pandas as pd

tiles_rows = []
for split in ("val", "test"):
    m = json.loads(Path(f"preds/slice/slice_meta_{split}.json").read_text())
    tiles_rows.append({"split": split, "device": m["device"],
                       "tiles/image": m.get("tiles_per_image_mean"),
                       "ms/image full view (hflip TTA)": m["ms_per_image"].get("full"),
                       "ms/image all tiles": m["ms_per_image"].get("tiles")})
print("Sliced inference wall-clock on this device (not the published CPU latency):")
display(pd.DataFrame(tiles_rows).set_index("split"))

casc_rows = []
for p in sorted(Path("preds").glob("cascade_*/cascade_meta_*_*.json")):
    c = json.loads(p.read_text())
    casc_rows.append({"dump": p.parent.name, "mode": c["mode"],
                      "dets scored": c["num_output"], "seconds": c["seconds"]})
if casc_rows:
    print()
    print("Crop refiner wall-clock:")
    display(pd.DataFrame(casc_rows))

pub = Path("viscan_beat_benchmark_results.zip")
members = []
for folder in list(Path("evaluation").rglob("*.json")) + list(Path("preds").rglob("*.json")):
    members.append((folder, str(folder)))
for extra in ("checkpoints/cascade/convnext_cascade.pth", "checkpoints/cascade/crop_bank_meta.json",
              "checkpoints/retinanet/best_model.pth", "runs/retinanet/loss_curve.png",
              "runs/retinanet/val_ap_curve.png"):
    if Path(extra).exists():
        members.append((Path(extra), extra))
for log in Path(".").glob("*.log"):
    members.append((log, log.name))

import zipfile
with zipfile.ZipFile(pub, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for src, arc in members:
        zf.write(src, arc)
print(f"zip: {pub.resolve()} ({pub.stat().st_size / 1024**2:.1f} MB, {len(members)} files)")

md_lines = ["| pipeline | val AP | val AP50 | test AP | test AP50 | test AP75 | test AR@100 | tau | test macro-F1 |",
            "|---|---|---|---|---|---|---|---|---|"]
for name, r in table.iterrows():
    md_lines.append("| " + name + " | " + " | ".join(f"{float(r[c]):.4f}" for c in
                    ["val AP", "val AP50", "test AP", "test AP50", "test AP75", "test AR@100", "tau", "test macro-F1"]) + " |")
md_table = "\n".join(md_lines)
Path("report_rows.md").write_text(md_table, encoding="utf-8")
print("\nMeasured rows for REPORT.md (paste only after reading them):\n")
print(md_table)

Sliced inference wall-clock on this device (not the published CPU latency):


,device,tiles/image,ms/image full view (hflip TTA),ms/image all tiles
split,,,,
val,cuda,11.37,437.44,1719.71
test,cuda,10.98,438.46,1668.21



Crop refiner wall-clock:


,dump,mode,dets scored,seconds
0,cascade_retinanet,reclass,7271,152.5
1,cascade_retinanet,rescore,12338,154.2
2,cascade_retinanet,reclass,10295,216.8
3,cascade_retinanet,rescore,17217,217.0
4,cascade_slice_full,reclass,3026,98.7
5,cascade_slice_full,rescore,3303,98.1
6,cascade_slice_full,reclass,4519,126.2
7,cascade_slice_full,rescore,4841,123.8
8,cascade_slice_tiles,reclass,8116,183.4
9,cascade_slice_tiles,rescore,14353,184.3


zip: /kaggle/working/viscan-opg/viscan_beat_benchmark_results.zip (367.5 MB, 80 files)

Measured rows for REPORT.md (paste only after reading them):

| pipeline | val AP | val AP50 | test AP | test AP50 | test AP75 | test AR@100 | tau | test macro-F1 |
|---|---|---|---|---|---|---|---|---|
| P0 Faster R-CNN | 0.3602 | 0.5705 | 0.3293 | 0.4961 | 0.3830 | 0.6834 | 0.7000 | 0.4016 |
| P1 FRCNN + hflip TTA | 0.3685 | 0.5773 | 0.3446 | 0.5281 | 0.3978 | 0.6852 | 0.8000 | 0.3988 |
| P2 FRCNN sliced (full+tiles) | 0.1960 | 0.3390 | 0.2189 | 0.3557 | 0.2521 | 0.6638 | 0.3500 | 0.2896 |
| P3 P2 + RetinaNet (WBF) | 0.1451 | 0.2553 | 0.1790 | 0.3015 | 0.1965 | 0.6462 | 0.3000 | 0.2661 |
| P4 P2 + cascade rescore | 0.2238 | 0.3908 | 0.2211 | 0.3589 | 0.2467 | 0.6560 | 0.4000 | 0.3543 |
| P5 P3 + cascade rescore | 0.1859 | 0.3257 | 0.1936 | 0.3168 | 0.2084 | 0.6613 | 0.4000 | 0.3172 |
| P6 P2 + cascade reclass | 0.1379 | 0.2476 | 0.1700 | 0.2715 | 0.1903 | 0.5573 | 0.2500 | 0.3184 |
| P7 P3 + casca

## Reading the result honestly

1. **Selection:** the pipeline chosen above comes from validation mAP@50:95 only. If its test numbers are worse
   than P0, that is the measured outcome — write it down as such, do not re-select on test.
2. **Reproduction:** P0 should land within ~0.002-0.003 of the published 0.496 / 0.329. A larger drift means the
   environment changed something (torchvision version, preprocessing, downloaded weights); stop and investigate
   before trusting any other row.
3. **Latency:** sliced inference multiplies forward passes by the tile count and the refiner adds a classifier
   pass per box. Even if AP improves, the deployed single-pass model stays the deployment default unless the
   gain justifies the cost. The numbers above are wall-clock on this GPU, not the published CPU latency.
4. **Comparability:** this test split has 50 images (182 boxes); the DENTEX leaderboard has 250 hidden images.
   Never present our AP50 as a leaderboard position.
5. **Paste policy:** copy the measured table into `REPORT.md`; mark which rows were measured in this session and
   which came from the earlier report. No number may be written before this run produces it.

In [17]:
os.chdir(WORKDIR)
import shutil

assert Path("viscan_beat_benchmark_results.zip").exists(), "build the zip before cleanup"
for p in ("data/dentex/zips", "data/dentex/extracted_train", "data/dentex/extracted_val"):
    target = Path(p)
    if target.exists():
        shutil.rmtree(target)
        print("removed", target)

print("final /kaggle/working size:",
      round(sum(f.stat().st_size for f in Path("/kaggle/working").rglob("*") if f.is_file()) / 1024**3, 2),
      "GB")

removed data/dentex/zips
removed data/dentex/extracted_train
removed data/dentex/extracted_val
final /kaggle/working size: 5.5 GB
